In [4]:
import torch
import pprint
lam_path: str = "/mnt/mnt/public/chenjl/code/UniVLA/latent_action_model/logs/vjepa_lam/epoch=14-step=20000.ckpt"

In [7]:
lam_ckpt = torch.load(lam_path, map_location="cpu")['state_dict']

for k, v in lam_ckpt.items():
    print(k)


lam.vision_encoder.encoder.patch_embed.proj.weight
lam.vision_encoder.encoder.patch_embed.proj.bias
lam.vision_encoder.encoder.blocks.0.norm1.weight
lam.vision_encoder.encoder.blocks.0.norm1.bias
lam.vision_encoder.encoder.blocks.0.attn.qkv.weight
lam.vision_encoder.encoder.blocks.0.attn.qkv.bias
lam.vision_encoder.encoder.blocks.0.attn.proj.weight
lam.vision_encoder.encoder.blocks.0.attn.proj.bias
lam.vision_encoder.encoder.blocks.0.norm2.weight
lam.vision_encoder.encoder.blocks.0.norm2.bias
lam.vision_encoder.encoder.blocks.0.mlp.fc1.weight
lam.vision_encoder.encoder.blocks.0.mlp.fc1.bias
lam.vision_encoder.encoder.blocks.0.mlp.fc2.weight
lam.vision_encoder.encoder.blocks.0.mlp.fc2.bias
lam.vision_encoder.encoder.blocks.1.norm1.weight
lam.vision_encoder.encoder.blocks.1.norm1.bias
lam.vision_encoder.encoder.blocks.1.attn.qkv.weight
lam.vision_encoder.encoder.blocks.1.attn.qkv.bias
lam.vision_encoder.encoder.blocks.1.attn.proj.weight
lam.vision_encoder.encoder.blocks.1.attn.proj.bias


In [1]:
import tensorflow

2025-09-19 13:18:11.903219: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-09-19 13:18:11.941236: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-09-19 13:18:11.941273: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-09-19 13:18:11.942145: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-09-19 13:18:11.947699: I tensorflow/core/platform/cpu_feature_guar

In [2]:
print(tensorflow.config.list_physical_devices('GPU'))

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]


In [2]:
!export HF_HOME="/mnt/public_zgc/home/jlchen/.cache/huggingface"
!export HF_HUB_CACHE="/mnt/public_zgc/home/jlchen/.cache/huggingface/models"
!export HF_DATASETS_CACHE="/mnt/public_zgc/home/jlchen/.cache/huggingface/datasets"
from transformers import AutoVideoProcessor, AutoModel

hf_repo = "facebook/vjepa2-vith-fpc64-256"

model = AutoModel.from_pretrained(hf_repo, force_download=True)
processor = AutoVideoProcessor.from_pretrained(hf_repo)


2025-09-22 12:46:04.810409: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-09-22 12:46:05.340166: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-09-22 12:46:05.340226: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-09-22 12:46:05.404927: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-09-22 12:46:05.566404: I tensorflow/core/platform/cpu_feature_guar

ValueError: Force download failed due to the above error.

In [4]:
import json
import os
from dataclasses import dataclass, field, asdict
from pathlib import Path
from typing import Optional, Tuple, Union, Dict, Any
from datetime import datetime

import torch
import torch.distributed as dist
import yaml
from latent_action_model.core.lam_model import load_latent_action_model
from prismatic.overwatch import initialize_overwatch
from prismatic.util import set_global_seed
from prismatic.vla import get_latent_vla_dataset_and_collator
from prismatic.vla.datasets.datasets import RLDSDataset
from prismatic.models import load_InternVL,freeze_internvl
from prismatic.vla.datasets.rlds.utils.data_utils import save_dataset_statistics
from typing import cast
from prismatic.training.accelerate_fsdp_trainer import run_latent_action_training
from transformers import AutoProcessor

# Sane Defaults
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")


# Initialize Overwatch =>> Wraps `logging.Logger`
overwatch = initialize_overwatch(__name__)

# 📝 动作 token 列表: ['<ACT_0>', '<ACT_1>', '<ACT_2>', '<ACT_3>', '<ACT_4>', '<ACT_5>', '<ACT_6>', '<ACT_7>', '<ACT_8>', '<ACT_9>', '<ACT_10>', '<ACT_11>', '<ACT_12>', '<ACT_13>', '<ACT_14>', '<ACT_15>']
# 🔢 对应的 token ID: [151679, 151680, 151681, 151682, 151683, 151684, 151685, 151686, 151687, 151688, 151689, 151690, 151691, 151692, 151693, 151694]
# 🎯 action_token_begin_id = 151679
# 📊 ID 范围: 151679 - 151694

@dataclass
class TrainConfig:
    # fmt: off

    # =========================
    # 路径与资源
    # =========================
    # Directory Paths
    data_root_dir: Path = Path("/mnt/public_zgc/home/jlchen/datasets")
    run_root_dir: Path =  Path("vla_log")    # Store logs & checkpoints under vla_scripts/

    # Hugging Face 模型标识（或本地权重目录）；用于 PrismaticVLM.from_pretrained()
    model_id: str = '/mnt/public_zgc/home/jlchen/weights/InternVL3_5-1B-Instruct-HF'
    hf_cache_dir: Optional[Path] = None
    lam_path: str = "/mnt/public_zgc/home/jlchen/code/UniVLA/latent_action_model/logs/version_1/checkpoints/epoch=40_step=41000.ckpt"

    # =========================
    # 数据与预处理
    # =========================
    # 数据混合与缓冲
    data_mix: str = "droid_100"
    shuffle_buffer_size: int = 20_00
    image_resolution: int = 448
    image_aug: bool = True                                          # Whether to enable image augmentations

    # =========================
    # 模型冻结策略
    # =========================
    # 冻结策略
    freeze_vision_backbone: bool = False
    freeze_llm_backbone: bool = False
    freeze_last_llm_layer: bool = False
    freeze_projector: bool = False

    # =========================
    # LAM / 动作离散参数
    # =========================
    action_token_begin_id: int = 151679
    # VJEPA_LAM 模型架构参数
    vision_model_id: str = "/mnt/public_zgc/home/jlchen/weights/vjepa2-vitl-fpc64-256"
    codebook_size: int = 16  #此处修改无效，仅作为标记
    # =========================
    # 训练设置
    # =========================
    # 训练超参
    epochs: Optional[int] = 10
    max_steps: Optional[int] = 200000  #以max_steps为准，若为空则按epochs * 10000近似
    per_device_batch_size: int = 4
    gradient_accumulation_steps: int = 1
    learning_rate: float = 2e-3
    warmup_steps: int = 1000
    weight_decay: float = 0.0
    max_grad_norm: float = 1.0
    lr_scheduler_type: str = "constant"   #constant_with_warmup
    # 训练加速
    enable_mixed_precision_training: bool = True
    seed: int = 42                                                  # Random seed (for reproducibility)

    # =========================
    # 评估与保存
    # =========================
    eval_strategy: str = "steps"
    save_total_limit: int = -1
    eval_interval: int = 4
    eval_accumulation_steps: int = 1
    per_device_eval_batch_size: int = 8
    save_interval: int = 1000                                    # Interval for saving checkpoints (in steps

    # =========================
    # 分布式 / FSDP
    # =========================
    fsdp: Optional[str] = "full_shard"                     # 示例："full_shard auto_wrap" 或 None 关闭
    fsdp_config: Optional[Dict[str, Any]] = None   # 示例：{"fsdp_min_num_params": 1e7, "xla": False}

    # =========================
    # 运行与日志
    # =========================
    # Run Arguments
    run_id: Optional[str] =  None                                  # Run ID for logging, Weights & Biases
    run_id_note: Optional[str] = "run_01"                               # Extra note for logging, Weights & Biases
    # Tracking Parameters
    wandb_project: str = "vla_pretraining"                   # Name of W&B project to log to (use default!)
    # wandb_entity: str = "opendrivelab"                              # Name of entity to log under

    # =========================
    # 恢复 / 断点（仅用于日志标记，不再用于模型权重加载）
    # =========================
    # Resume (logging) Parameters -- 仅用于日志标记，不再用于模型权重加载
    resume_step: Optional[int] = None
    resume_epoch: Optional[int] = None

    # =========================
    # HF Hub 凭据（如有门限模型）
    # =========================
    # HF Hub Credentials (for any gated models)
    hf_token: Optional[str] = None

    # fmt: on

cfg=TrainConfig

overwatch.info("OpenVLA Training :: Warming Up")

# Note => Under `torchrun` initializing `overwatch` will automatically set up `torch.distributed`
torch.cuda.set_device(device_id := overwatch.local_rank())
torch.cuda.empty_cache()

# Configure Unique Run Name & Save Directory
vla_tag = f"{cfg.model_id.split('/')[-1]}+{cfg.data_mix}"
world_size = overwatch.world_size() if dist.is_initialized() else max(torch.cuda.device_count(), 1)
overwatch.info(f"Detected world_size = {world_size}")
cfg.run_id = (
    f"{vla_tag}+n{world_size}+b{cfg.per_device_batch_size}+x{cfg.seed}"
    if cfg.run_id is None
    else cfg.run_id
)
if cfg.run_id_note is not None:
    cfg.run_id += f"--{cfg.run_id_note}"
if cfg.image_aug:
    cfg.run_id += "--image_aug"

# cfg.run_id += '-Latent-Action-Pretraining'
# Start =>> Build Directories and Set Randomness
overwatch.info('"Do or do not; there is no try."', ctx_level=1)
# hf_token = cfg.hf_token.read_text().strip() if isinstance(cfg.hf_token, Path) else os.environ[cfg.hf_token]
hf_token = str(cfg.hf_token) if isinstance(cfg.hf_token, Path) else cfg.hf_token
worker_init_fn = set_global_seed(cfg.seed, get_worker_init_fn=True)
# 统一时间戳并广播，确保所有进程共用同一目录
if dist.is_initialized():
    if overwatch.is_rank_zero():
        ts: Optional[str] = datetime.now().strftime("%m%d_%H%M%S")
    else:
        ts = None
    obj_list = [ts]
    dist.broadcast_object_list(obj_list, src=0)
    timestamp = obj_list[0]
    assert isinstance(timestamp, str)
else:
    timestamp = datetime.now().strftime("%m%d_%H%M%S")

run_dir_name = f"{timestamp}+{cfg.run_id}"
run_dir = (cfg.run_root_dir / run_dir_name)
# 仅 rank0 创建目录，其余进程等待
if (not dist.is_initialized()) or overwatch.is_rank_zero():
    os.makedirs(run_dir, exist_ok=True)
    try:
        os.makedirs(run_dir / "checkpoints", exist_ok=True)
    except Exception:
        pass
if dist.is_initialized():
    dist.barrier()

# 仅 rank0 写入文件日志，并避免重复添加 FileHandler
if (not dist.is_initialized()) or overwatch.is_rank_zero():
    try:
        import logging

        log_path = str(run_dir / "train.log")
        root_logger = logging.getLogger()
        already_attached = False
        for h in list(root_logger.handlers):
            try:
                if hasattr(h, "baseFilename") and getattr(h, "baseFilename") == log_path:
                    already_attached = True
                    break
            except Exception:
                continue
        if not already_attached:
            file_handler = logging.FileHandler(log_path, mode="a", encoding="utf-8")
            formatter = logging.Formatter("| >> %(message)s", datefmt="%m/%d [%H:%M:%S]")
            file_handler.setFormatter(formatter)
            root_logger.addHandler(file_handler)
    except Exception:
        pass

# os.makedirs(cfg.run_root_dir / cfg.run_id / "checkpoints", exist_ok=True)



# 直接通过 HF ID/Path 加载 InternVL 模型与处理器
overwatch.info(f"🔄 加载基础 InternVL `{cfg.model_id}`（HF from_pretrained）")
vlm, tokenizer = load_InternVL(cfg.model_id, cfg.hf_cache_dir, dtype=torch.bfloat16) 
vlm.generation_config.max_new_tokens = int(getattr(cfg, "max_new_tokens", 4))
vlm.generation_config.pad_token_id=int(tokenizer.eos_token_id)
vlm.config.loss_type = str(getattr(cfg, "loss_type", "ForCausalLMLoss"))
vlm.config.use_cache = False

# 直接按配置冻结模块（若可用）；HF-only InternVL 组件名：vision_tower / language_model / multi_modal_projector / lm_head
freeze_internvl(vlm, cfg.freeze_vision_backbone, cfg.freeze_projector, cfg.freeze_llm_backbone, cfg.freeze_last_llm_layer)
overwatch.info("🔧 扩充 LLM 词表以注入动作离散 token")
special_tokens_dict = {'additional_special_tokens': [f'<ACT_{i}>' for i in range(cfg.codebook_size)]}
try:
    num_added_toks = tokenizer.add_special_tokens(special_tokens_dict)  # type: ignore[attr-defined]
    overwatch.info(f"num_added_toks={num_added_toks}")
except Exception:
    num_added_toks = 0

# Print number of total/trainable model parameters
num_params = sum(p.numel() for p in vlm.parameters())
num_trainable_params = sum(p.numel() for p in vlm.parameters() if p.requires_grad)
overwatch.info(
    f"# Parameters (in millions): {num_params / 10**6:.3f} Total, {num_trainable_params / 10**6:.3f} Trainable"
)

# Get VLA Dataset & Collator

overwatch.info(
    f"🔄 加载 V-JEPA2 动作编码器与码本（ckpt=`{cfg.lam_path}`，"
    f"K={cfg.codebook_size}）"
)
latent_action_model = load_latent_action_model(cfg.lam_path, vision_model_id=cfg.vision_model_id)  # default freeze all parameters
latent_action_model = latent_action_model.to(device_id).eval()
overwatch.info(
    f"🔄 构建 RLDS 数据集与 Collator（mixture=`{cfg.data_mix}`，image_res={cfg.image_resolution}）"
)
# 类型提示规避：latent_action_tokenizer 需要 VQ 编码器，这里用 cast 静态规避
train_dataset, val_dataset, tokenizer, collator = get_latent_vla_dataset_and_collator(
    cfg.data_root_dir,
    cfg.data_mix,
    latent_action_model,
    tokenizer=tokenizer,
    default_image_resolution=cfg.image_resolution,
    shuffle_buffer_size=cfg.shuffle_buffer_size,
    image_aug=cfg.image_aug,
)
   

[*] OpenVLA Training :: Warming Up
[*] Detected world_size = 2
    |=> "Do or do not; there is no try."
[*] 🔄 加载基础 InternVL `/mnt/public_zgc/home/jlchen/weights/InternVL3_5-1B-Instruct-HF`（HF from_pretrained）
[*] 🔧 扩充 LLM 词表以注入动作离散 token
[*] num_added_toks=16
[*] # Parameters (in millions): 1060.898 Total, 1060.898 Trainable
[*] 🔄 加载 V-JEPA2 动作编码器与码本（ckpt=`/mnt/public_zgc/home/jlchen/code/UniVLA/latent_action_model/logs/version_1/checkpoints/epoch=40_step=41000.ckpt`，K=16）
[*] 🔄 构建 RLDS 数据集与 Collator（mixture=`droid_100`，image_res=448）
Load dataset info from /mnt/public_zgc/home/jlchen/datasets/droid_100/1.0.0
Constructing tf.data.Dataset r2d2_faceblur for split all, from /mnt/public_zgc/home/jlchen/datasets/droid_100/1.0.0
2025-09-27 12:29:21.906597: I tensorflow/core/grappler/optimizers/data/replicate_on_split.cc:32] Running replicate on split optimization
[*] Loading existing dataset statistics from /mnt/public_zgc/home/jlchen/datasets/droid_100/1.0.0/dataset_statistics_e4f1ab21815de

cannot find val split, use train[:95%]

######################################################################################
# Loading the following 1 datasets (incl. sampling weight):                         #
# droid_100: ===============================================================1.000000 #
######################################################################################

cannot find val split, use train[:95%]


Constructing tf.data.Dataset r2d2_faceblur for split train[:95%], from /mnt/public_zgc/home/jlchen/datasets/droid_100/1.0.0
2025-09-27 12:29:22.272395: I tensorflow/core/grappler/optimizers/data/replicate_on_split.cc:32] Running replicate on split optimization
[*] Applying frame transforms on dataset...
Load dataset info from /mnt/public_zgc/home/jlchen/datasets/droid_100/1.0.0
Constructing tf.data.Dataset r2d2_faceblur for split all, from /mnt/public_zgc/home/jlchen/datasets/droid_100/1.0.0
2025-09-27 12:29:23.086600: I tensorflow/core/grappler/optimizers/data/replicate_on_split.cc:32] Running replicate on split optimization
[*] Loading existing dataset statistics from /mnt/public_zgc/home/jlchen/datasets/droid_100/1.0.0/dataset_statistics_e4f1ab21815dedb79156d90a4688cb938333262b42e97441f245fe95d20eae4b.json.
Constructing tf.data.Dataset r2d2_faceblur for split train[95%:], from /mnt/public_zgc/home/jlchen/datasets/droid_100/1.0.0
2025-09-27 12:29:23.427565: I tensorflow/core/grappler

cannot find val split, use train[:95%]

######################################################################################
# Loading the following 1 datasets (incl. sampling weight):                         #
# droid_100: ===============================================================1.000000 #
######################################################################################

cannot find val split, use train[:95%]


Constructing tf.data.Dataset r2d2_faceblur for split train[95%:], from /mnt/public_zgc/home/jlchen/datasets/droid_100/1.0.0
2025-09-27 12:29:23.614490: I tensorflow/core/grappler/optimizers/data/replicate_on_split.cc:32] Running replicate on split optimization
[*] Applying frame transforms on dataset...


In [ ]:
from torch.utils.data import DataLoader
val_data = DataLoader(
            train_dataset,
            collate_fn=collator,
            num_workers=0,
        )
# 获取一个批次的数据
batch = next(iter(val_data))

print("=== 批次数据结构 ===")
for key, value in batch.items():
    if isinstance(value, torch.Tensor):
        print(f"{key}: {value.shape} | dtype: {value.dtype}")
    elif isinstance(value, list):
        print(f"{key}: List[{len(value)}] | 类型: {type(value[0]) if value else 'Empty'}")
    else:
        print(f"{key}: {type(value)}")

W0000 00:00:1758947368.265371  508097 op_level_cost_estimator.cc:699] Error in PredictCost() for the op: op: "CropAndResize" attr { key: "T" value { type: DT_FLOAT } } attr { key: "extrapolation_value" value { f: 0 } } attr { key: "method" value { s: "bilinear" } } inputs { dtype: DT_FLOAT shape { dim { size: 1 } dim { size: 448 } dim { size: 448 } dim { size: -14 } } } inputs { dtype: DT_FLOAT shape { dim { size: -2 } dim { size: 4 } } } inputs { dtype: DT_INT32 shape { dim { size: -2 } } } inputs { dtype: DT_INT32 shape { dim { size: 2 } } } device { type: "CPU" vendor: "GenuineIntel" model: "106" frequency: 3000 num_cores: 28 environment { key: "cpu_instruction_set" value: "AVX SSE, SSE2, SSE3, SSSE3, SSE4.1, SSE4.2" } environment { key: "eigen" value: "3.4.90" } l1_cache_size: 49152 l2_cache_size: 1310720 l3_cache_size: 50331648 memory_size: 268435456 } outputs { dtype: DT_FLOAT shape { dim { size: -2 } dim { size: -16 } dim { size: -17 } dim { size: -14 } } }
W0000 00:00:175894736

=== 批次数据结构 ===
pixel_values: torch.Size([1, 3, 448, 448]) | dtype: torch.float32
input_ids: torch.Size([1, 350]) | dtype: torch.int64
attention_mask: torch.Size([1, 350]) | dtype: torch.bool
labels: torch.Size([1, 350]) | dtype: torch.int64


In [ ]:

for i in range(20):
    data = next(iter(val_data))
    print(data['labels'][data["labels"] != -100])

W0000 00:00:1758947606.249824  508097 op_level_cost_estimator.cc:699] Error in PredictCost() for the op: op: "CropAndResize" attr { key: "T" value { type: DT_FLOAT } } attr { key: "extrapolation_value" value { f: 0 } } attr { key: "method" value { s: "bilinear" } } inputs { dtype: DT_FLOAT shape { dim { size: 1 } dim { size: 448 } dim { size: 448 } dim { size: -14 } } } inputs { dtype: DT_FLOAT shape { dim { size: -2 } dim { size: 4 } } } inputs { dtype: DT_INT32 shape { dim { size: -2 } } } inputs { dtype: DT_INT32 shape { dim { size: 2 } } } device { type: "CPU" vendor: "GenuineIntel" model: "106" frequency: 3000 num_cores: 28 environment { key: "cpu_instruction_set" value: "AVX SSE, SSE2, SSE3, SSSE3, SSE4.1, SSE4.2" } environment { key: "eigen" value: "3.4.90" } l1_cache_size: 49152 l2_cache_size: 1310720 l3_cache_size: 50331648 memory_size: 268435456 } outputs { dtype: DT_FLOAT shape { dim { size: -2 } dim { size: -16 } dim { size: -17 } dim { size: -14 } } }
W0000 00:00:175894760

tensor([151687, 151690, 151690, 151690])


W0000 00:00:1758947607.092531  508097 op_level_cost_estimator.cc:699] Error in PredictCost() for the op: op: "CropAndResize" attr { key: "T" value { type: DT_FLOAT } } attr { key: "extrapolation_value" value { f: 0 } } attr { key: "method" value { s: "bilinear" } } inputs { dtype: DT_FLOAT shape { dim { size: 1 } dim { size: 448 } dim { size: 448 } dim { size: -14 } } } inputs { dtype: DT_FLOAT shape { dim { size: -2 } dim { size: 4 } } } inputs { dtype: DT_INT32 shape { dim { size: -2 } } } inputs { dtype: DT_INT32 shape { dim { size: 2 } } } device { type: "CPU" vendor: "GenuineIntel" model: "106" frequency: 3000 num_cores: 28 environment { key: "cpu_instruction_set" value: "AVX SSE, SSE2, SSE3, SSSE3, SSE4.1, SSE4.2" } environment { key: "eigen" value: "3.4.90" } l1_cache_size: 49152 l2_cache_size: 1310720 l3_cache_size: 50331648 memory_size: 268435456 } outputs { dtype: DT_FLOAT shape { dim { size: -2 } dim { size: -16 } dim { size: -17 } dim { size: -14 } } }
W0000 00:00:175894760

tensor([151692, 151692, 151692, 151692])


W0000 00:00:1758947607.773447  508097 op_level_cost_estimator.cc:699] Error in PredictCost() for the op: op: "CropAndResize" attr { key: "T" value { type: DT_FLOAT } } attr { key: "extrapolation_value" value { f: 0 } } attr { key: "method" value { s: "bilinear" } } inputs { dtype: DT_FLOAT shape { dim { size: 1 } dim { size: 448 } dim { size: 448 } dim { size: -14 } } } inputs { dtype: DT_FLOAT shape { dim { size: -2 } dim { size: 4 } } } inputs { dtype: DT_INT32 shape { dim { size: -2 } } } inputs { dtype: DT_INT32 shape { dim { size: 2 } } } device { type: "CPU" vendor: "GenuineIntel" model: "106" frequency: 3000 num_cores: 28 environment { key: "cpu_instruction_set" value: "AVX SSE, SSE2, SSE3, SSSE3, SSE4.1, SSE4.2" } environment { key: "eigen" value: "3.4.90" } l1_cache_size: 49152 l2_cache_size: 1310720 l3_cache_size: 50331648 memory_size: 268435456 } outputs { dtype: DT_FLOAT shape { dim { size: -2 } dim { size: -16 } dim { size: -17 } dim { size: -14 } } }
W0000 00:00:175894760

...

tensor([], dtype=torch.int64)


W0000 00:00:1758947630.678814  508097 op_level_cost_estimator.cc:699] Error in PredictCost() for the op: op: "CropAndResize" attr { key: "T" value { type: DT_FLOAT } } attr { key: "extrapolation_value" value { f: 0 } } attr { key: "method" value { s: "bilinear" } } inputs { dtype: DT_FLOAT shape { dim { size: 1 } dim { size: 448 } dim { size: 448 } dim { size: -14 } } } inputs { dtype: DT_FLOAT shape { dim { size: -2 } dim { size: 4 } } } inputs { dtype: DT_INT32 shape { dim { size: -2 } } } inputs { dtype: DT_INT32 shape { dim { size: 2 } } } device { type: "CPU" vendor: "GenuineIntel" model: "106" frequency: 3000 num_cores: 28 environment { key: "cpu_instruction_set" value: "AVX SSE, SSE2, SSE3, SSSE3, SSE4.1, SSE4.2" } environment { key: "eigen" value: "3.4.90" } l1_cache_size: 49152 l2_cache_size: 1310720 l3_cache_size: 50331648 memory_size: 268435456 } outputs { dtype: DT_FLOAT shape { dim { size: -2 } dim { size: -16 } dim { size: -17 } dim { size: -14 } } }
W0000 00:00:175894763

tensor([151679, 151679, 151679, 151679])


W0000 00:00:1758947631.430295  508097 op_level_cost_estimator.cc:699] Error in PredictCost() for the op: op: "CropAndResize" attr { key: "T" value { type: DT_FLOAT } } attr { key: "extrapolation_value" value { f: 0 } } attr { key: "method" value { s: "bilinear" } } inputs { dtype: DT_FLOAT shape { dim { size: 1 } dim { size: 448 } dim { size: 448 } dim { size: -14 } } } inputs { dtype: DT_FLOAT shape { dim { size: -2 } dim { size: 4 } } } inputs { dtype: DT_INT32 shape { dim { size: -2 } } } inputs { dtype: DT_INT32 shape { dim { size: 2 } } } device { type: "CPU" vendor: "GenuineIntel" model: "106" frequency: 3000 num_cores: 28 environment { key: "cpu_instruction_set" value: "AVX SSE, SSE2, SSE3, SSSE3, SSE4.1, SSE4.2" } environment { key: "eigen" value: "3.4.90" } l1_cache_size: 49152 l2_cache_size: 1310720 l3_cache_size: 50331648 memory_size: 268435456 } outputs { dtype: DT_FLOAT shape { dim { size: -2 } dim { size: -16 } dim { size: -17 } dim { size: -14 } } }
W0000 00:00:175894763

KeyboardInterrupt: 